In [ ]:
# jupyter nbconvert "data/<your_input_file>" --no-input --to html

# Deep sequencing analysis for GS BTHC Batch1

# QC module

## 1 Functions and module

### 1.1 Modules

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import gridspec
import numpy as np
import math
import seaborn as sns
import regex
import scipy.stats

In [ ]:
pd.set_option('display.max_columns', 30)

### 1.2 Functions

In [ ]:
def generate_raw_dataframe(
    read_path,
    grna_path,
    experiment_path,
    barcode_pattern=None
):
    """
    Load read counts, gRNA annotation, and experiment metadata.
    Clean and merge them into a unified dataframe.
    """

    # ----------------------------------------------------------
    # 1. Load read count table
    # ----------------------------------------------------------
    read_df = pd.read_csv(read_path)
    read_df = read_df.rename(columns={'Frequency': 'Count'})

    # ----------------------------------------------------------
    # 2. Load gRNA annotation table
    # ----------------------------------------------------------
    grna_df = pd.read_csv(grna_path)
    grna_df = grna_df.rename(columns={
        'Gene': 'Targeted_gene_name',
        'sgRNA': 'gRNA'
    })

    # Drop duplicate rows safely
    grna_df = grna_df.drop_duplicates()

    # gRNA vs Spikein identity
    grna_df['Identity'] = (
        grna_df['Targeted_gene_name']
        .apply(lambda x: 'Spikein' if 'Spike' in x else 'gRNA')
    )

    # Numbered gene names for duplicated genes
    grna_df['Numbered_gene_name'] = grna_df['Targeted_gene_name']
    duplicated_mask = grna_df['Numbered_gene_name'].duplicated(keep=False)

    grna_df.loc[duplicated_mask, 'Numbered_gene_name'] += (
        '_' + grna_df.groupby('Numbered_gene_name').cumcount().add(1).astype(str)
    )

    # ----------------------------------------------------------
    # 3. Load experiment metadata
    # ----------------------------------------------------------
    exp_df = pd.read_csv(experiment_path)

    exp_df = exp_df.rename(columns={
        'Sample ID': 'Sample_ID',
        'Mouse_Genotype': 'Mouse_genotype',
        'Virus_Titer': 'Virus_titer',
        'Time_After_Tumor_Initiation(wks)': 'Time_after_tumor_initiation',
        'Total_Lung_Weight(g)': 'Total_lung_weight'
    })

    # ----------------------------------------------------------
    # 4. Combine all
    # ----------------------------------------------------------
    output_df = combine_dataframe_Info(read_df, grna_df, exp_df)

    # ----------------------------------------------------------
    # 5. Optional: Barcode pattern filtering
    # ----------------------------------------------------------
    if barcode_pattern:
        pattern = regex.compile(barcode_pattern)

        # Pre-assign default
        output_df['Saturation_barcode'] = 'Spikein_barcode'

        # Vectorized pattern checking for gRNA rows
        is_grna = output_df['Identity'] == 'gRNA'

        output_df.loc[is_grna, 'Saturation_barcode'] = (
            output_df.loc[is_grna, 'Clonal_barcode']
            .apply(lambda bc: 'Real' if pattern.search(bc) else 'Fake')
        )

        # Remove invalid barcodes
        output_df = output_df[output_df['Saturation_barcode'] != 'Fake']

    # ----------------------------------------------------------
    # 6. Reporting
    # ----------------------------------------------------------
    total_reads = read_df['Count'].sum()
    kept_reads = output_df['Count'].sum()

    print(f"There are totally {total_reads:,} reads.")
    print(f"There are {kept_reads:,} reads mapped to non-spikein gRNA with expected barcode pattern "
          f"({kept_reads / total_reads:.3%}).")

    return output_df


In [ ]:
def combine_dataframe_Info(input_reads_df, input_gRNA_df, input_sample_df):
    # this function take reads, gRNA info df and sample df, and merge to a final data df
    # merge reads df with gRNA info df. Only gRNA match with reference will be selected
    temp_df = input_reads_df.merge(
        input_gRNA_df, how='inner', on=['gRNA'], sort=True)
    # merge reads df with sample info df
    output_df = temp_df.merge(
        input_sample_df, how='inner', on=['Sample_ID'], sort=True)
    output_df['gRNA_clonalbarcode'] = output_df['gRNA'] + '_' + output_df['Clonal_barcode']
    return (output_df)

In [ ]:
def generate_sample_summary(input_df, input_spikein_check_df,input_cell_number_cutoff,input_read_cutoff):
    # input_cell_number_cutoff is the cell number cutoff
    # input_rad_cutoff is the read cutoff
    # Total reads does not restricted to gRNA or cell number cutoff
    temp_df0 = input_df.groupby(['Sample_ID','Mouse_Ear_Tag','Mouse_genotype', 'Sex','Pooling_library_name',
         'Time_after_tumor_initiation', 'Total_lung_weight', 'Virus_titer','Correction_for_spikein','Cell_number_per_read','Tissue_type'],as_index=False).agg(
        TTR = pd.NamedAgg('Count',aggfunc = sum))
    temp_df0 = temp_df0.merge(input_spikein_check_df[['Sample_ID','Mean_count','max_least_ratio']],on = 'Sample_ID') # merge spike in info
    temp_df0['Spikein_read_ratio'] = (temp_df0['Mean_count']*3)/temp_df0['TTR']
    # filter input data
    temp_input = input_df[(input_df['Cell_number']>=input_cell_number_cutoff)&(input_df['Count']>input_read_cutoff)]
    # I only consider non-spikein gRNA
    temp_df1 = temp_input[temp_input['Identity']=='gRNA'].groupby(
        ['Sample_ID'],as_index = False).apply(
        cal_sample_summary)
    temp_df1['TTB_million'] = temp_df1['TTB']/1000000
    # merge sample and gRNA information    
    temp_df1 = temp_df1.merge(temp_df0,on = 'Sample_ID',how = 'right')
    # normalize to per 100K virus
    temp_df1['Tumor number per 100K virus'] = temp_df1.apply(lambda x: x['TTN']/x['Virus_titer']*100000,axis=1)
    temp_df1['Total tumor burden (million per 100K virus)'] = np.log10(temp_df1.apply(lambda x: x['TTB_million']/x['Virus_titer']*100000,axis=1))
    return(temp_df1)

In [ ]:
def spikein_summary(input_df,input_spikein_name,input_cell_number):
# this function extract spike reads for each kind in each sample
    temp_df = input_df[input_df['Targeted_gene_name'].isin(input_spikein_name)] # select spike in rows
    temp_df = temp_df.groupby(['Sample_ID','Targeted_gene_name'],as_index = False).agg(
        Unique_spikein_barcode_number = pd.NamedAgg('Clonal_barcode',aggfunc = lambda x: len(x)), # number of different spike-in barcode
        Count = pd.NamedAgg('Count',aggfunc = np.sum)
    )
    temp_df['Cell_per_read'] = input_cell_number/temp_df['Count']
    temp_df['Amplification'] = temp_df.Count/temp_df.Unique_spikein_barcode_number
    return(temp_df)

In [ ]:
# calculate the combined spikein metrics for each sample
def cal_spikein_ratio(x,spikein_name_list):
    d = {}
    temp_vect = x[['Targeted_gene_name','Count']]
    s1_value = x[x['Targeted_gene_name'] == spikein_name_list[0]]['Count'].values[0]
    s2_value = x[x['Targeted_gene_name'] == spikein_name_list[1]]['Count'].values[0]
    s3_value = x[x['Targeted_gene_name'] == spikein_name_list[2]]['Count'].values[0]
    d['Spikein1'] = s1_value
    d['Spikein2'] = s2_value
    d['Spikein3'] = s3_value
    d['Mean_count'] = (s1_value+s2_value+s3_value)/3 # mean count for spike in 
    s1_s2_ratio =  s1_value/s2_value# spikein 1/ spikein 2 ratio
    s1_s3_ratio =  s1_value/s3_value# spikein 1/ spikein 2 ratio
    d['s1_s2_ratio'] = s1_s2_ratio
    d['s1_s3_ratio'] = s1_s3_ratio
    d['max_least_ratio'] = sorted([s1_value,s2_value,s3_value])[-1]/sorted([s1_value,s2_value,s3_value])[0]
    return pd.Series(d, index=list(d.keys())) 

In [ ]:
def label_point(x, y, val, ax):
    a = pd.concat({'x': x, 'y': y, 'val': val}, axis=1)
    for i, point in a.iterrows():
        ax.text(point['x']+.02, point['y'], str(point['val']),size = 8)

In [ ]:
def generate_simple_sample_summary(input_df,input_spikein_df):
    temp_df0 = input_df.groupby(['Sample_ID','Mouse_genotype','Virus_titer','Total_lung_weight','Mouse_Ear_Tag','Pooling_library_name'],as_index=False).agg(
        TTR = pd.NamedAgg('Count',aggfunc = sum)) # total reads
    spikein_name_list = ['tuba-seq-v2_Spike-in-1','tuba-seq-v2_Spike-in-2','tuba-seq-v2_Spike-in-3'] # spike in name should be ordered here
    temp_spike_df = input_spikein_df.set_index('Sample_ID')
    temp_df0['Spike1_reads_fraction'] = temp_spike_df[temp_spike_df['Targeted_gene_name']==spikein_name_list[0]].loc[temp_df0.Sample_ID]['Count'].to_list()/temp_df0.TTR
    temp_df0['Spike2_reads_fraction'] = temp_spike_df[temp_spike_df['Targeted_gene_name']==spikein_name_list[1]].loc[temp_df0.Sample_ID]['Count'].to_list()/temp_df0.TTR
    temp_df0['Spike3_reads_fraction'] = temp_spike_df[temp_spike_df['Targeted_gene_name']==spikein_name_list[2]].loc[temp_df0.Sample_ID]['Count'].to_list()/temp_df0.TTR
    return(temp_df0)

In [ ]:
def generate_conversion_factor(input_spike_df, input_candidate_list, input_spike_in_list,temp_spikein_cell_number):
    # this function generate a df storing cell number per reads
    # input_spike_df is the df contain the spikein information (spikein_ratio_df)
    # temp_spikein_cell_number is the cell number for each spike in added into sample (usually it is 100K)
    # for most sample, I will use temp_spikein_cell_number/ mean(spike in reads)
    # input_candidate_list is the list of sample id whose spikein needed special treatment
    # temp_spikein_cell_number specify which spike in will be use for each sample ID who need special treatment
    input_dic = dict(zip(input_candidate_list, input_spike_in_list))
    temp_dic = {}
    for index,row in input_spike_df.iterrows():
        temp_id = row['Sample_ID']
        if temp_id in input_dic.keys():
            # print(row[input_dic.get(id)])
            temp_value = row[input_dic.get(temp_id)].mean()
        else:
            temp_value = row['Mean_count']
        temp_dic[temp_id] = temp_spikein_cell_number/temp_value
    temp_df = pd.DataFrame({'Sample_ID':temp_dic.keys(),
                       'Cell_number_per_read':temp_dic.values()})
    temp_df['Correction_for_spikein'] = temp_df['Sample_ID'].apply(lambda x: 'Yes' if (x in input_candidate_list) else 'No')
    return(temp_df.sort_values(by=['Correction_for_spikein','Cell_number_per_read']))
        

In [ ]:
def generate_final_df(input_df,input_conversion_factor_df):
    # input_df is the dataframe input
    # input_conversion_factor_df is the dictionary that store the the cell number per reads
    temp_df = input_df.merge(input_conversion_factor_df, on ='Sample_ID')
    temp_df['Cell_number'] = temp_df['Count']*temp_df['Cell_number_per_read']
    return(temp_df)

In [ ]:
# calculate the summary metrics for each sample
def cal_sample_summary(x):
    d = {}
    temp_vect = x['Cell_number']
    if type (temp_vect) == 'int':
        temp_vect = [temp_vect]
    d['gRNA_recovered'] = len(x['gRNA'].unique())
    d['TTB'] = sum(temp_vect) # total mutational burdern 
    d['TTN'] = len(temp_vect) # this is total tumor number
    return pd.Series(d, index=list(d.keys())) 

In [ ]:
def filter_and_aggregate(df, n):
    """
    For each Sample_ID:
      - Select the top `n` rows ranked by TTB.
      - Aggregate all remaining rows into a single "others" entry.
    Returns a new DataFrame with top genes + aggregated 'others'.
    """

    aggregated_results = []

    for sample_id, group in df.groupby('Sample_ID'):
        # Top n by TTB
        top_rows = group.nlargest(n, 'TTB')

        # Aggregate remaining rows into "others"
        if len(group) > n:
            remaining = group.iloc[n:]

            others_row = pd.DataFrame({
                'Sample_ID': [sample_id],
                'Targeted_gene_name': ['others'],
                'gRNA': ['others'],
                'Mouse_genotype': [group['Mouse_genotype'].iloc[0]],  # consistent within sample
                'TTN': [remaining['TTN'].sum()],
                'TTB': [remaining['TTB'].sum()],
            })

            top_rows = pd.concat([top_rows, others_row], ignore_index=True)

        aggregated_results.append(top_rows)

    # Combine all samples together
    return pd.concat(aggregated_results, ignore_index=True)


----

## 2 Input and output address

In [ ]:
# combined barcode dataframe address
parent_address = "data/"
combined_df_address = parent_address + "gRNA_clonalbarcode_combined.csv"
# gRNA information address
gRNA_info_address = parent_address + "gRNA_information.csv"
# experimental information address
exp_info_address = parent_address + "GS_BTHC_Batch1_mice_info_standardized.csv"
# library pooling info addfress 
# lib_info_address = parent_address + 'GW_Library_info_standardized.csv'

In [ ]:
project_prefix = 'GS_BTHC_Batch1'
sample_summary_address = parent_address + f"{project_prefix}_sample_summary_df.csv"
annotated_data_output_address = parent_address + f"{project_prefix}_annotated_df.parquet"
fig_output_address = parent_address + f"{project_prefix}_QC_fig.pdf"

final_data_output_address = parent_address + f"{project_prefix}_final_df.parquet"

-----

## 3 Raw data processing

### 3.1 Explaination and note
* <font size="5"> 3.2 only gRNAs matched to existing gRNA or spike in are retained</font>
* <font size="5"> 3.3 I want to know if any spikein has unexpected high or low read in certain sample</font>
* <font size="5"> 3.2 I only consider barcode pattern that matched the Jackie ID pattern</font>

### 3.2 raw data Generation

In [ ]:
raw_summary_df = generate_raw_dataframe(combined_df_address,gRNA_info_address,exp_info_address,barcode_pattern='(A.{14}G)')

In [ ]:
raw_summary_df.head()

In [ ]:
k = raw_summary_df[['Sample_ID','Mouse_genotype']].drop_duplicates()
k['Mouse_genotype'].value_counts()

## 4 Spikein QC

### 4.1 Count spikein read numbers and ratio

In [ ]:
# I first check if spikes have similar read counts 
spikein_name_list = ['tuba-seq-v2_Spike-in-1','tuba-seq-v2_Spike-in-2','tuba-seq-v2_Spike-in-3'] # spike in name should be ordered here
spikein_df = spikein_summary(raw_summary_df,spikein_name_list,50000)

In [ ]:
spikein_df.head()

In [ ]:
spikein_df.shape

In [ ]:
spikein_ratio_df = spikein_df.groupby(['Sample_ID'],as_index = False).apply(cal_spikein_ratio,spikein_name_list) # the ratio of spike in

In [ ]:
spikein_ratio_df.head()

### 4.2 Find samples with unusual spike in ratio

In [ ]:
plt.scatter(spikein_ratio_df.s1_s2_ratio,spikein_ratio_df.s1_s3_ratio)
label_point(spikein_ratio_df.s1_s2_ratio,spikein_ratio_df.s1_s3_ratio, spikein_ratio_df['Sample_ID'], plt.gca())
# plt.xscale('log',base = 2)
# plt.yscale('log',base = 2)
plt.xlabel('s1/s2')
plt.ylabel('s1/s3')

* <font size="5" color =  black> All looks good</font>

### 4.3 Find bad spike in for those samples

#### 4.3.1 Method 1

In [ ]:
input_ratio_cutoff = 1.4 # this is the cutoff 
temp_bad_list = spikein_ratio_df.loc[spikein_ratio_df['max_least_ratio']>input_ratio_cutoff,'Sample_ID'].to_list() # list of sample with bad spikein

In [ ]:
# check the max/min ratio in each sample and print those sample with ratio above certain cutoff
fig1,ax = plt.subplots(math.ceil(len(temp_bad_list)/4), 4, figsize=(20,5*math.ceil(len(temp_bad_list)/4)))
axes = ax.flatten()
for x,y in zip(axes[:len(temp_bad_list)],temp_bad_list):
    temp_df = spikein_df[spikein_df['Sample_ID'] == y]
    x.bar(temp_df['Targeted_gene_name'],temp_df['Count'])
    x.set_title(y)
    x.set_xticklabels(['spike_in_1','spike_in_2','spike_in_3'])
for x in axes[len(temp_bad_list):]:
    x.set_axis_off()
fig1.text(0.08, 0.5, 'Count', va='center', rotation='vertical',fontsize=20)
fig1.text(0.5, 0.01, 'Spikein', ha='center',fontsize=20)
fig1.text(0.5, 0.95, 'Samples with unexpected spike-in ratio (>{})'.format(input_ratio_cutoff), ha='center',fontsize=20)
# fig1.savefig(figure_output_address+'QC_332.pdf')aa

* <font size="5" color =  black> All samples are very good</font>

#### 4.3.2 Method 2

In [ ]:
temp_summary_df = generate_simple_sample_summary(raw_summary_df,spikein_df)

In [ ]:
temp_summary_df.head()

In [ ]:
gs = gridspec.GridSpec(1, 17) 
fig1 = plt.figure(figsize=(17,5))
ax1=fig1.add_subplot(gs[:1, 0:5])
temp_df = temp_summary_df
ix = 'Mouse_genotype'
iy = 'Spike1_reads_fraction'
# sns.scatterplot(data=temp_df, x=ix,y =iy ,ax = ax1,hue = 'Pooling_library_name',)
sns.boxplot(data=temp_df, x=ix,y =iy ,hue = 'Pooling_library_name', ax = ax1)
# ax1.set_yscale('log', base=10)

In [ ]:
temp_df.sort_values(by='Spike1_reads_fraction')

In [ ]:
temp_df.groupby('Mouse_genotype', as_index=False)['Spike1_reads_fraction'].mean()

In [ ]:
temp_df.groupby('Mouse_genotype', as_index=False)['Spike2_reads_fraction'].mean()

In [ ]:
temp_df.groupby('Mouse_genotype', as_index=False)['Spike3_reads_fraction'].mean()

In [ ]:
# Define variables to plot
spike_fractions = ['Spike1_reads_fraction', 'Spike2_reads_fraction', 'Spike3_reads_fraction']

# Filter data
temp_df = temp_summary_df[temp_summary_df.Spike1_reads_fraction<0.1]

# Find global axis limits
x_min, x_max = temp_df['Total_lung_weight'].min(), temp_df['Total_lung_weight'].max()
y_min = min(temp_df[spike_fractions].min())  # Get min across all y-axes
y_max = max(temp_df[spike_fractions].max())  # Get max across all y-axes

# Add buffer
x_buffer = (x_max - x_min) * 0.05
y_buffer = (y_max - y_min) * 0.02

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)

for i, iy in enumerate(spike_fractions):
    ax = axes[i]
    
    # Regression plot
    sns.regplot(data=temp_df, x='Total_lung_weight', y=iy, ax=ax)
    
    # Highlight bad samples
    temp_sub = temp_df[temp_df.Sample_ID.isin(temp_bad_list)]
    ax.scatter(temp_sub['Total_lung_weight'], temp_sub[iy], color='red', label='Bad Samples', zorder=3)
    
    # Annotate points
    for x, y, label in zip(temp_sub['Total_lung_weight'], temp_sub[iy], temp_sub['Sample_ID']):
        ax.text(x, y, label, fontsize=8, ha='right')

    ax.set_title(f"{iy}")
    ax.set_xlabel('Total Lung Weight')
    if i == 0:
        ax.set_ylabel('Spike Reads Fraction')
    
    # **Manually adjust limits**
    ax.set_xlim(x_min - x_buffer, x_max + x_buffer)
    ax.set_ylim(y_min - y_buffer, y_max + y_buffer)

# Adjust layout
plt.tight_layout()
plt.show()


* <font size="8" color = "red"> The idea here is that if some spike-in read number is very distort, I will throw it away</font>

In [ ]:
temp = [['Spikein2','Spikein3']]
temp_bad = []
conversion_factor_df= generate_conversion_factor(spikein_ratio_df,temp_bad,temp,50000)

In [ ]:
conversion_factor_df.sort_values(by='Cell_number_per_read').head()

## 5 Output annotate data

### 5.1 Anotate data

In [ ]:
annotated_df = generate_final_df(raw_summary_df,conversion_factor_df)

### 5.3 Output annotated_df

In [ ]:
annotated_df.to_parquet(annotated_data_output_address,index=False)